# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset


In [3]:
df = pd.read_csv("cleaned_airbnb_data_7.csv")
df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,log_price,availability_scaled,reviews_standardized,review_activity,price_per_guest,long_term_stay,room_type_Entire home/apt,room_type_Hotel room,room_type_Private room,room_type_Shared room
0,934859141300145599,https://www.airbnb.com/rooms/934859141300145599,20260622055616,2026-07-02,previous scrape,An Exquisite Rooftop Studio,Introducing an exquisite rooftop studio in the...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,384892315,...,4.127134,0.997260,-0.535855,Low,61.000000,0,1,0,0,0
1,944156894640610027,https://www.airbnb.com/rooms/944156894640610027,20260622055616,2026-06-23,city scrape,Tarrytown bungalow,Take a break and unwind at this peaceful oasis...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,13392053,...,5.010635,0.986301,-0.498194,Low,74.500000,0,0,0,1,0
2,1634902892490996951,https://www.airbnb.com/rooms/1634902892490996951,20260622055616,2026-06-23,city scrape,Central East Gem near running trail,Keep it simple at this peaceful and centrally-...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,266329439,...,6.226537,1.000000,-0.535855,Low,84.166667,0,1,0,0,0
3,1308737991593278409,https://www.airbnb.com/rooms/1308737991593278409,20260622055616,2026-06-23,city scrape,Empire,"- Accommodate up to 14 guests, 4-bedroom home<...",NaN,https://a0.muscache.com/pictures/miso/Hosting-...,398748229,...,6.386879,0.928767,-0.149831,Medium,42.357143,0,1,0,0,0
4,574582741829434703,https://www.airbnb.com/rooms/574582741829434703,20260622055616,2026-06-23,city scrape,Galaxy by AvantStay | Luxury Amenities in Austin,- Pet-friendly home<br />- Rooftop pool and lo...,NaN,https://a0.muscache.com/pictures/prohost-api/H...,434308464,...,5.633610,0.978082,-0.243983,Medium,69.667500,0,1,0,0,0


### Reflection:
1. What business information does the dataset include?
2. How many rows and columns are present?

### ✍️ Your Response: 🔧
1. The dataset includes the listing prices, the room type, availability, review standardized and activity.

2. There are 11,295 rows and 96 columns present.

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

In [6]:
df = df.drop(columns=["id", "host_id", "name", "description"], errors="ignore")

### Reflection:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?

### ✍️ Your Response: 🔧
1. The columns I dropped were the id, host_id, name, and desciprtion because those don't really give any useful information, they're just descriptions.

2. If I kept them, it would be unecessary and might cause random noise or problems that have to do with accuracy when it comes to modeling.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

In [7]:
corr = df.corr(numeric_only=True)
print(corr["price"].sort_values(ascending=False))

price                          1.000000
price_quote_price_per_night    1.000000
price_per_guest                0.575808
price_quote_total_price        0.537717
log_price                      0.520248
                                 ...   
host_acceptance_rate                NaN
host_neighbourhood                  NaN
host_total_listings_count           NaN
neighbourhood                       NaN
instant_bookable                    NaN
Name: price, Length: 66, dtype: float64


### Reflection:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?

### ✍️ Your Response: 🔧
1. The bathrooms had the strongest positive correlation, the long_term_stay had a negative correlation with prive.

2. Bathrooms, bedrooms, accomodates, and beds might be useful predictors.

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

In [9]:
features = ["accommodates", "bathrooms", "bedrooms", "beds",
            "minimum_nights", "availability_365", "number_of_reviews",
            "review_scores_rating", "latitude", "longitude",
            "long_term_stay", "room_type_Entire home/apt",
            "room_type_Hotel room", "room_type_Private room"]

data = df[features + ["price"]].dropna()
X = data[features]
y = data["price"]

### Reflection:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?

### ✍️ Your Response: 🔧
1. I used the bathrooms, bedrooms, beds, availability, reviews, location, and room types.

2. This is a regression problem and not a classification problem because it's continuous and numerical, not a category like yes/no.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [11]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

In [12]:
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("R²:", r2)

MSE: 1883133.7143883405
R²: 0.11602664928662931


### Reflection:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?

### ✍️ Your Response: 🔧
1. My R² score was 0.1160 so the model explained about 11.6% of the price variation which is not very well..

2. My MSE is 1,883,133.7143 which is actually really large. I could probably improve it by handling extreme prices or trying more predictors.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

In [13]:
coef = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

coef.reindex(coef["Coefficient"].abs().sort_values(ascending=False).index)

,Feature,Coefficient
9,longitude,-598.922951
12,room_type_Hotel room,482.096098
1,bathrooms,280.575149
8,latitude,173.222472
13,room_type_Private room,83.160062
10,long_term_stay,67.111949
11,room_type_Entire home/apt,59.109049
2,bedrooms,-26.393890
0,accommodates,15.400580
3,beds,-15.191701


### Reflection:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?

### ✍️ Your Response: 🔧
1. It looks like hotel room and bathrooms increased price the most as those are in the positives.

2. I see that longitude is super negative which is surprising because it shows that the location has something to do with the price.

3. From the results, I think we could do better and figure out how location influences pricing. The model doesn't determine causation, but it'd be good to look into.


## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

In [14]:
top = coef.loc[coef["Coefficient"].abs().nlargest(5).index, "Feature"]

model2 = LinearRegression()
model2.fit(X_train[top], y_train)
pred2 = model2.predict(X_test[top])

print("MSE:", mean_squared_error(y_test, pred2))
print("R²:", r2_score(y_test, pred2))

MSE: 1864660.3417168858
R²: 0.12469834849449302


### Reflection:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?

### ✍️ Your Response: 🔧
1. In the refined model, I kept the hotel room, bathrooms, longtitude, latitude, and private room because they had the strongest relationships in the model.

2. The model did improve a bit because the R² is now 0.125 when it was originally 0.1160. I think removing some varibales helped.

3. To stakeholders, I would recommend using the refined model to start as a baseline because it's better than the original model. But both still need more improvements.

4. This relates to my customized learning outcomes of building and evaluating models to help support financial decisions. I was able to improve a model and see how adding and removing predictors can help make a model more accurate.


## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### Reflection:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. My refined model helped answer which Airbnb listing was associated with price the most and which one of those can be used to predict listing prices in Austin, Texas.

2. Although my model still needs improvement and is somewhat limited, I would recommend to Airbnb and its hosts to consider the bathrooms, room types, location, etc when it comes to their pricing.

3. To improve this model, I would look into the extreme prices, add and test more predictors, and maybe try other model methods besides regression.

4. This relates to my customized learning outcomes that I created on canvas because I was able to build and evaluate models to forecast pricing and financial decisions. I was able to find the MSE and R², compare and improve models.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_11_regression.ipynb"